# Arcora Week 1 — Exploratory Data Analysis on Historical NBA Dataset

This notebook explores Arcora's canonical NBA datasets (`games.parquet`, `player_game_stats.parquet`, `players.parquet`, `teams.parquet`) across 20 NBA seasons (`2003-04` through `2022-23`).

In [1]:
import pandas as pd
from arcora.config import ArcoraConfig
from arcora.eda.analysis import run_exploratory_data_analysis

cfg = ArcoraConfig.from_env()
proc = cfg.processed_data_dir
games_df = pd.read_parquet(proc / 'games.parquet')
pstats_df = pd.read_parquet(proc / 'player_game_stats.parquet')
players_df = pd.read_parquet(proc / 'players.parquet')
teams_df = pd.read_parquet(proc / 'teams.parquet')
print(f'Games: {len(games_df):,}, Player-Game Stats: {len(pstats_df):,}, '
      f'Players: {len(players_df):,}, Teams: {len(teams_df):,}')

Games: 26,523, Player-Game Stats: 668,338, Players: 2,687, Teams: 30


## 1. Scoring & Point-Margin Distributions

We examine the distributions of `home_pts`, `away_pts`, `total_pts`, and `point_margin` (`home_pts - away_pts`).

In [2]:
summary_cols = ['home_pts', 'away_pts', 'total_pts', 'point_margin']
print(games_df[summary_cols].describe().round(2))

       home_pts  away_pts  total_pts  point_margin
count  26523.00  26523.00   26523.00      26523.00
mean     103.45    100.63     204.08          2.82
std       13.28     13.43      23.04         13.51
min       36.00     33.00      69.00        -68.00
25%       94.00     91.00     188.00         -7.00
50%      103.00    100.00     203.00          4.00
75%      112.00    110.00     219.00         11.00
max      168.00    168.00     329.00         73.00


## 2. Historical Season Trends & Home-Court Advantage (2003–2022)

Below we analyze game counts, scoring evolution, and home-court win rates across all 20 seasons, including lockout (`2011-12`) and COVID-19 (`2019-20`, `2020-21`) anomalies.

In [3]:
season_summary = games_df.groupby('season_label').agg(
    games=('game_id', 'count'),
    mean_home_pts=('home_pts', 'mean'),
    mean_away_pts=('away_pts', 'mean'),
    mean_total_pts=('total_pts', 'mean'),
    home_win_rate=('home_win', 'mean'),
).round(3)
print(season_summary)

              games  mean_home_pts  mean_away_pts  mean_total_pts  \
season_label                                                        
2003-04        1286         94.907         91.093         186.000   
2004-05        1362         98.605         95.382         193.987   
2005-06        1432         98.406         95.172         193.579   
2006-07        1419         99.849         96.944         196.794   
2007-08        1411        101.274         97.549         198.823   
2008-09        1425        100.923         97.904         198.827   
2009-10        1424        101.654         98.746         200.400   
2010-11        1422        100.757         97.385         198.141   
2011-12        1104         97.392         94.429         191.822   
2012-13        1420         99.382         96.114         195.496   
2013-14        1427        101.735         99.261         200.996   
2014-15        1418        101.095         98.815         199.910   
2015-16        1416        103.713

## 3. Extreme Game Results & Correlations

We inspect extreme historical games (highest/lowest combined scoring and largest home/away blowouts) and box-score differential correlations.

In [4]:
eda_res = run_exploratory_data_analysis(proc, cfg.figures_dir)
for ex in eda_res.extreme_games:
    print(ex.category, ex.game_date, ex.away_team, ex.home_team)
print('Correlations:', eda_res.correlations_with_point_margin)

Highest Combined Total Points 2019-03-01 CHI ATL
Lowest Combined Total Points 2007-10-19 BKN BOS
Largest Home Victory Margin 2021-12-02 OKC MEM
Largest Away Victory Margin 2021-10-13 DAL CHA
Highest Single-Game Home Team Score 2008-03-16 OKC DEN
Highest Single-Game Away Team Score 2019-03-01 CHI ATL
Correlations: {'fg_pct_diff': 0.7717, 'fg3_pct_diff': 0.5085, 'ft_pct_diff': 0.172, 'reb_diff': 0.4559, 'ast_diff': 0.6012}


## 4. Candidate Predictive Features & Modeling Takeaways

- **Lagged Rolling Efficiency & Margin**: 5-game and 10-game rolling net rating and effective FG% differential.
- **Dynamic Elo Ratings**: Season-spanning team strength with home-court offset.
- **Rest & Schedule Density**: Days of rest and back-to-back indicators.
- **Era-Adjusted Pace & 3-Point Volume**: Accounts for the +39 PPG scoring surge between 2003-04 and 2022-23.